# Keystroke-based user verification

Each row of the dataset contains timing measurements from one repetition of a fixed phrase. After **column-wise scaling** and **centering**, we form the sample covariance matrix, perform **PCA** via eigen-decomposition, and classify new samples by the **Euclidean norm** of the projection relative to a threshold **τ**.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import keystroke as ks
import plots

FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

## 1. Input data

The CSV stores `subject`, session metadata, and numeric dwell/flight-style timings (31 features per row for this file).

In [ ]:
csv_path = ks.DATASET_CSV
peek = pd.read_csv(csv_path, nrows=6)
display(peek.iloc[:, :10])

## 2. Train / holdout evaluation

One subject is treated as **genuine**; other subjects supply **impostor** trials. The model is fit on a random subset of genuine rows; τ is chosen on holdout genuine vs impostor score distributions.

In [ ]:
summary = ks.run_dataset_demo(n_components=12, plot_path=None)
figure_paths = plots.save_report_figures(summary, FIG_DIR)
print("Saved", len(figure_paths), "figures under", FIG_DIR)

## 3. Principal components

Eigenvalues of the covariance matrix (after scaling and subtracting the mean) show how much variance each axis captures.

In [ ]:
for path in [FIG_DIR / "01_pca_eigenvalues.png", FIG_DIR / "02_variance_explained.png"]:
    display(Image(filename=str(path)))

## 4. Verification scores

The classifier uses $\|P^T(x_{scaled} - \mu)\|$. Genuine and impostor holdout scores should separate around τ.

In [ ]:
for path in [FIG_DIR / "03_genuine_vs_impostor_distances.png", FIG_DIR / "04_training_distances.png"]:
    display(Image(filename=str(path)))

## 5. Pipeline and holdout accuracy

End-to-end flow and empirical correct-decision rates on held-out rows.

In [ ]:
for path in [
    FIG_DIR / "05_pipeline_schematic.png",
    FIG_DIR / "06_holdout_accuracy.png",
    FIG_DIR / "07_summary_panel.png",
]:
    display(Image(filename=str(path)))